# ICT4403 Assignment 3 — English to Hindi Translation
**Yash Shukla · 210911380**

This notebook contains the full source project. It trains a basic Transformer, fine-tunes mT5, prompts LLaMA and Gemma, evaluates one shared test set, compares sampling temperatures and creates the submission documents.

**Before starting:** obtain access using your own Hugging Face account to [LLaMA](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct) and [Gemma](https://huggingface.co/google/gemma-3-1b-it). Review their terms yourself. Keep your read token private.

Run cells in order. Model cells can take substantial time. A 16 GB GPU is a practical starting point, not a guaranteed runtime or memory budget. No results are prefilled in this notebook. All metrics come from executions. The separately supplied CPU basic-model baseline is a failed partial experiment, not the completed assignment.

## 1 · Save the project in Google Drive
Choose **Runtime → Change runtime type → GPU** first. Mount Drive when prompted so files survive a disconnect. Use a new `RUN_FOLDER` if changing the experiment. This cell writes code, but preserves existing experiment results.

In [ ]:
from google.colab import drive
from pathlib import Path
import base64, io, os, zipfile

drive.mount('/content/drive')
RUN_FOLDER = 'FGAI_Assignment3_GPU_Run'
PROJECT = Path('/content/drive/MyDrive') / RUN_FOLDER
PROJECT.mkdir(parents=True, exist_ok=True)
PAYLOAD = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as archive:
    for name in archive.namelist():
        dest = PROJECT / name
        if dest.exists() and (name == 'config.json' or name.startswith('results/')):
            continue
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(archive.read(name))
os.chdir(PROJECT)
print('Project:', PROJECT)
print('Code is ready. Existing results and configuration were preserved.')


## 2 · Install the pinned dependencies
Wait for completion. If Colab explicitly asks you to restart the session, do so, rerun cell 1 and continue. Do not run the install cell repeatedly after a successful installation.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


## 3 · Check GPU and model access
The token entry is hidden. If an access check fails, get approval on that model page for the same account before proceeding. Never place tokens in visible notebook cells or outputs.

In [ ]:
import torch
from huggingface_hub import login, hf_hub_download
from getpass import getpass
assert torch.cuda.is_available(), 'Select a GPU in Runtime > Change runtime type, then reconnect.'
print('GPU:', torch.cuda.get_device_name(0))
try:
    from google.colab import userdata
    hf_read_token = userdata.get('HF_TOKEN')
except Exception:
    hf_read_token = getpass('Paste your Hugging Face READ token (hidden): ')
login(token=hf_read_token, add_to_git_credential=False)
del hf_read_token
for model_name in ['meta-llama/Llama-3.2-1B-Instruct', 'google/gemma-3-1b-it']:
    hf_hub_download(model_name, 'config.json')
    print('Access confirmed:', model_name)


## 4 · Run component checks and prepare data
The dataset is downloaded from IIT Bombay's Hugging Face distribution. Official train/validation/test partitions remain separate. The saved source revision and split hashes make the subset reproducible.

In [ ]:
import subprocess, sys, json
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)
subprocess.run([sys.executable, 'run.py', '--prepare-only'], check=True)
print(Path('results/data_manifest.json').read_text())


## 5 · Train the basic Transformer
This cell runs the model, scores the shared test set, saves generated translations, runs the temperature experiment and prepares ethical-review files. Do not interrupt a running cell. Training checkpoint files remain private in Drive.

In [ ]:
subprocess.run([sys.executable, 'run.py', '--models', 'basic'], check=True)


## 6 · Fine-tune mT5
This cell runs the model, scores the shared test set, saves generated translations, runs the temperature experiment and prepares ethical-review files. Do not interrupt a running cell. Training checkpoint files remain private in Drive.

In [ ]:
subprocess.run([sys.executable, 'run.py', '--models', 'mt5'], check=True)


## 7 · Prompt LLaMA
This cell runs the model, scores the shared test set, saves generated translations, runs the temperature experiment and prepares ethical-review files. Do not interrupt a running cell. Training checkpoint files remain private in Drive.

In [ ]:
subprocess.run([sys.executable, 'run.py', '--models', 'llama'], check=True)


## 8 · Prompt Gemma
This cell runs the model, scores the shared test set, saves generated translations, runs the temperature experiment and prepares ethical-review files. Do not interrupt a running cell. Training checkpoint files remain private in Drive.

In [ ]:
subprocess.run([sys.executable, 'run.py', '--models', 'gemma'], check=True)


## 9 · See the measured comparison and actual translations
A blank or poor translation remains a scored failure. Small overlap scores do not prove useful translation. Inspect the output text and the responsible-AI probes.

In [ ]:
import pandas as pd
from IPython.display import display, Image, Markdown
from src.summarize import summarize
summarize(Path.cwd())
display(pd.read_csv('results/metrics.csv'))
for name in ['basic','mt5','llama','gemma']:
    print('MODEL:', name)
    display(pd.read_csv(f'results/{name}_translations.csv').head(10).fillna(''))
    display(pd.read_csv(f'results/{name}_ethical_probes.csv').fillna(''))
for path in ['quality_comparison.png','compute_comparison.png','temperature_analysis.png']:
    display(Image(filename='results/'+path))
display(Markdown(Path('ETHICS.md').read_text()))


## 10 · Download and fill the review worksheets
Download this ZIP, extract it, and fill the scores/comments in a spreadsheet editor. For each of the four models, rate at least the **first 10 test rows** and **all 8 probes**. Save as UTF-8 CSV with the original filenames. Read `ETHICS.md` for the rubric. Notes must describe actual output, including omissions, stereotypes, changed facts or preserved meaning.

In [ ]:
from google.colab import files
with zipfile.ZipFile('Translation_Review_Worksheets.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in ['basic','mt5','llama','gemma']:
        for suffix in ['human_review','ethical_probes']:
            p=Path(f'results/{name}_{suffix}.csv')
            archive.write(p, p.name)
files.download('Translation_Review_Worksheets.zip')


## 11 · Upload the completed review CSVs
Select the eight edited CSV files, not the ZIP. This cell accepts only the expected filenames and stores them in the project.

In [ ]:
allowed = {f'{name}_{suffix}.csv' for name in ['basic','mt5','llama','gemma']
           for suffix in ['human_review','ethical_probes']}
review_uploads = files.upload()
for filename, content in review_uploads.items():
    if filename not in allowed:
        raise ValueError('Unexpected filename: '+filename+'. Keep the original CSV filenames.')
    Path('results', filename).write_bytes(content)
print('Saved', len(review_uploads), 'review worksheets.')


## 12 · Add the real GitHub link and generate the submission
Create a repository in your GitHub account named `fgai-assignment3-en-hi-translation`. Copy its actual URL into the hidden-free text prompt below. This script checks the format, not whether it has been published. Upload the final project after downloading it in the next step. The finalization check refuses a completed submission if model runs or review ratings are missing.

In [ ]:
REPO_URL = input('Paste the actual GitHub repository URL: ').strip()
subprocess.run([sys.executable, 'finalize.py', '--repo-url', REPO_URL], check=True)
print(Path('submission/readiness.json').read_text())


## 13 · Download the completed project
Extract the ZIP and upload its contents to GitHub. It excludes dataset copies, large checkpoints, caches and tokens. Separately download this executed notebook through **File → Download → Download .ipynb**, check that its outputs contain no secrets, and upload it too. Finally upload the personalized PDF from `submission/` to your LMS. The draft included in the original package must be replaced by this completed version.

In [ ]:
from src.common import read
assert read('submission/readiness.json')['ready_for_student_review'], 'Finish the previous steps first.'
archive_path = Path('/content/FGAI_Assignment3_Completed.zip')
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for p in Path.cwd().rglob('*'):
        if not p.is_file():
            continue
        rel=p.relative_to(Path.cwd())
        if any(part in {'checkpoints','data','__pycache__','.git','.cache','.ipynb_checkpoints'} for part in rel.parts):
            continue
        if p.suffix in {'.zip','.token'} or p.name == '.env':
            continue
        archive.write(p, str(rel))
files.download(str(archive_path))
files.download('submission/Yash_Shukla_210911380_FGAI_Assignment3.pdf')
